In [1]:
import numpy as np
import pandas as pd



In [2]:
df=pd.read_csv(r"/Users/siddhant/Downloads/credit_risk_dataset.csv")

In [3]:
df.head()

,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Education_Level,Housing_Status,Default
0,59,52154.0,11276,823,15,Bachelors,Own,0
1,49,116646.0,43663,315,5,PhD,Own,0
2,35,61157.0,18994,428,8,Masters,Own,1
3,63,52154.0,28499,408,26,Bachelors,Rent,0
4,28,148876.0,28040,832,3,Masters,Own,1


In [4]:
df.shape

(1000, 8)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               1000 non-null   int64  
 1   Income            985 non-null    float64
 2   Loan_Amount       1000 non-null   int64  
 3   Credit_Score      1000 non-null   int64  
 4   Employment_Years  1000 non-null   int64  
 5   Education_Level   1000 non-null   object 
 6   Housing_Status    1000 non-null   object 
 7   Default           1000 non-null   int64  
dtypes: float64(1), int64(5), object(2)
memory usage: 62.6+ KB


In [6]:
df.isnull().sum()/df.shape[0]*100

Age                 0.0
Income              1.5
Loan_Amount         0.0
Credit_Score        0.0
Employment_Years    0.0
Education_Level     0.0
Housing_Status      0.0
Default             0.0
dtype: float64

In [7]:
# pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)

## Separate Cat and Num cols

In [8]:
df_num_clm=df.select_dtypes(include=['int', 'float'])

In [9]:
df_cat_clm=df.select_dtypes(include=['object'])

## Handling num cols

In [10]:
df_num_clm.isnull().sum()

Age                  0
Income              15
Loan_Amount          0
Credit_Score         0
Employment_Years     0
Default              0
dtype: int64

In [11]:
df_num_clm['Income'].replace(np.nan, df_num_clm['Income'].median(), inplace= True )

/var/folders/9d/x9v4fczs01xdjhbh4msz_f_c0000gn/T/ipykernel_1447/1911211623.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_num_clm['Income'].replace(np.nan, df_num_clm['Income'].median(), inplace= True )


In [12]:
df_num_clm.isnull().sum()

Age                 0
Income              0
Loan_Amount         0
Credit_Score        0
Employment_Years    0
Default             0
dtype: int64

## Handling cat cols

In [13]:
df_cat_clm.isnull().sum()

Education_Level    0
Housing_Status     0
dtype: int64

In [14]:
cat_list=df_cat_clm.columns

In [15]:
cat_list

Index(['Education_Level', 'Housing_Status'], dtype='object')

### Encoding 

In [16]:
from sklearn.preprocessing import OneHotEncoder

In [17]:
ohe=OneHotEncoder(sparse_output=False)

In [18]:
df['Education_Level'].value_counts()

Education_Level
PhD            260
Masters        253
Bachelors      248
High School    239
Name: count, dtype: int64

In [19]:
df['Housing_Status'].value_counts()

Housing_Status
Mortgage    349
Own         330
Rent        321
Name: count, dtype: int64

In [20]:
for i in cat_list:
    df_cat_clm=pd.get_dummies(df_cat_clm,columns=[i],drop_first=True,dtype=int)

In [21]:
df_cat_clm.shape

(1000, 5)

### concat both cat and num cols

In [22]:
df_new=pd.concat([df_cat_clm,df_num_clm], axis=1)

In [23]:
df_new.head()

,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Default
0,0,0,0,1,0,59,52154.0,11276,823,15,0
1,0,0,1,1,0,49,116646.0,43663,315,5,0
2,0,1,0,1,0,35,61157.0,18994,428,8,1
3,0,0,0,0,1,63,52154.0,28499,408,26,0
4,0,1,0,1,0,28,148876.0,28040,832,3,1


In [24]:
from sklearn.model_selection import train_test_split

In [25]:
X = df_new.drop("Default", axis=1)
y = df_new["Default"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=12)

### Scaling

In [26]:
from sklearn.preprocessing import StandardScaler

ss=StandardScaler()

X_num_scaled=ss.fit_transform(X)
df2_scale=pd.DataFrame(X_num_scaled)
df3_scale=pd.DataFrame(df2_scale,columns= cat_list)

In [28]:
df_new

,Education_Level_High School,Education_Level_Masters,Education_Level_PhD,Housing_Status_Own,Housing_Status_Rent,Age,Income,Loan_Amount,Credit_Score,Employment_Years,Default
0,0,0,0,1,0,59,52154.0,11276,823,15,0
1,0,0,1,1,0,49,116646.0,43663,315,5,0
2,0,1,0,1,0,35,61157.0,18994,428,8,1
3,0,0,0,0,1,63,52154.0,28499,408,26,0
4,0,1,0,1,0,28,148876.0,28040,832,3,1
...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,1,0,1,53,44519.0,7307,433,22,1
996,1,0,0,1,0,22,107487.0,44901,582,7,1
997,0,1,0,0,1,34,102870.0,16205,372,29,0
998,0,0,1,1,0,60,66197.0,10906,780,24,0


## Logistic Regression

In [29]:
from sklearn.linear_model import LogisticRegression

In [30]:
lr=LogisticRegression(class_weight='balanced')

In [31]:
lr.fit(X_train,y_train)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [32]:
y_pred=lr.predict(X_test)

In [33]:
lr.score(X_test,y_pred)

1.0

In [34]:
from sklearn.metrics import precision_score, recall_score,f1_score, confusion_matrix, classification_report

In [35]:
print('P_score', precision_score(y_test, y_pred))

P_score 0.14814814814814814


In [36]:
print('R_score', recall_score(y_test, y_pred))

R_score 0.41379310344827586


In [37]:
print('f1_score', f1_score(y_test, y_pred))

f1_score 0.21818181818181817


In [38]:
print(confusion_matrix(y_test, y_pred))

[[102  69]
 [ 17  12]]


In [39]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.60      0.70       171
           1       0.15      0.41      0.22        29

    accuracy                           0.57       200
   macro avg       0.50      0.51      0.46       200
weighted avg       0.75      0.57      0.63       200

